In [ ]:
from moabb.datasets import *
from moabb.paradigms import P300
import tensorly as tl

tmin=-0.2
dataset = BNCI2014_008()
paradigm = P300(tmin=-0.2)
epochs, y, meta = paradigm.get_data(dataset, return_epochs=True)
X = epochs.get_data()
groups=meta['subject']
X = tl.tensor(X)
X.shape

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from hoda.classification import SelectFCutoff
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

clf = make_pipeline(
    FunctionTransformer(tl.to_numpy),
    PCA(n_components=None, whiten=True),
    SelectFCutoff(cutoff=1),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from hoda.classification import BTTDACV
from hoda.hoda import BTTDA

cv = StratifiedGroupKFold(n_splits=5)

hoda_params=  dict(
    toeplitz=(1,),
    verbose=True,
)


bttdacv_params = dict(
    hoda_params=hoda_params,
    verbose=True,
    cv=cv,
    n_jobs=-1,
    clf = clf,
)

bttdacv = BTTDACV(
    max_n_blocks=2,
    fixed_n_blocks=True,
    thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6, 0.7, 0.8, 0.9,1],    
    **bttdacv_params
)

    

In [ ]:
import joblib
from joblib import Parallel, delayed
import distributed
from hpc import create_cluster, create_client, TIMEOUT


with create_cluster(cluster='wice_sapphirerapids') as cluster, create_client(cluster) as client:
    with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
        client.wait_for_workers(5*11)
        bttdacv.fit(X,y, groups=groups)



In [ ]:
bttdacv.blocks_[0].theta

In [ ]:
bttdacv.blocks_[1].theta

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import plotly.express as px
from mne import EvokedArray
%matplotlib inline

plot_kwargs=dict(
    ts_args=dict(
        #ylim=dict(eeg=[-2,2.5])
    )
)

target = X[y=='Target'].mean(axis=0)
non_target = X[y=='NonTarget'].mean(axis=0)
contrast = target - non_target
evoked = EvokedArray(tl.to_numpy(contrast), epochs.info, tmin=tmin)
evoked = evoked.apply_baseline((None, 0))
evoked.save(f'results/interpretability/erp_contrast_ave.fif')
evoked.plot_joint(**plot_kwargs)

X_prev = tl.zeros_like(X)
for bi,b in enumerate(bttdacv.blocks_):
    Xt = bttdacv.transform(X, n_blocks=bi+1)
    Xr = bttdacv.inv_transform(Xt, n_blocks=bi+1) - X_prev
    X_prev = Xr
    
    target = Xr[y=='Target'].mean(axis=0)
    non_target = Xr[y=='NonTarget'].mean(axis=0)
    contrast = target - non_target
    evoked = EvokedArray(tl.to_numpy(contrast), epochs.info, tmin=tmin)
    evoked = evoked.apply_baseline((None,0))
    evoked.save(f'results/interpretability/erp_contrast_block-{bi}_ave.fif')
    evoked.plot_joint(**plot_kwargs)